# Serve the Misconception Tutor model (free GPU notebook)

This notebook loads **your** fine-tuned Gemma adapter and exposes it as an OpenAI-compatible API, so the
web app can call it. Run the cells top to bottom (about 5-10 minutes the first time), then copy the two values printed
by the last cell into your app's settings: `LLM_BACKEND_URL` and `LLM_BACKEND_TOKEN`.

**Before you start**
- Turn on a **GPU** (T4 or better) and **Internet** for this notebook.
- This is a temporary host: when the session ends (idle timeout, quota, or you close it) the app shows "model offline" until you run it again. Start it ~15 minutes before a demo.
- Kaggle is the notebook host your project already uses. Reports in 2026 say Colab's free tier may disallow hosting web servers/tunnels - check the platform's terms before relying on it.

In [ ]:
import os, subprocess, sys

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

sh("pip install -q -U unsloth flask requests")
if not os.path.isdir("/tmp/app"):
    sh("GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/kunalsinghparmar13/Misconception-Tutor.git /tmp/app")

## Where are the adapter weights?
Option A (recommended): upload the adapter once to the Hugging Face Hub with `scripts/push_adapter_to_hf.py`, then put the repo id below.
Option B: leave `HF_ADAPTER_ID` empty and the notebook pulls the 139 MB file from GitHub LFS (GitHub's free LFS bandwidth is limited).

In [ ]:
HF_ADAPTER_ID = ""   # e.g. "your-username/misconception-tutor-gemma4-e4b"
HF_TOKEN = ""        # only needed if that Hub repo is private

if HF_ADAPTER_ID:
    ADAPTER = HF_ADAPTER_ID
else:
    sh("apt-get -qq install -y git-lfs > /dev/null 2>&1 || true")
    sh("cd /tmp/app && git lfs install && git lfs pull --include='LLM/adapter_model.safetensors'")
    ADAPTER = "/tmp/app/LLM"
    size = os.path.getsize(ADAPTER + "/adapter_model.safetensors")
    assert size > 1_000_000, "adapter_model.safetensors is still a Git LFS pointer - use Option A instead."
print("Adapter:", ADAPTER)

In [ ]:
import secrets
TOKEN = secrets.token_urlsafe(24)
os.environ.update(MODEL_SERVER_TOKEN=TOKEN, LLM_ADAPTER_PATH=ADAPTER, LLM_GEN_BACKEND="unsloth")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

sys.path.insert(0, "/tmp/app/LLM")
from serve import model_server

model_server.load_model()   # same Unsloth loading code as tutor-project.ipynb
print("Model loaded.")

In [ ]:
import threading, requests
from werkzeug.serving import make_server

server = make_server("127.0.0.1", 8001, model_server.app, threaded=True)
threading.Thread(target=server.serve_forever, daemon=True).start()

r = requests.post(
    "http://127.0.0.1:8001/v1/chat/completions",
    headers={"Authorization": f"Bearer {TOKEN}"},
    json={"messages": [{"role": "user", "content": "A student says 1/4 + 1/6 = 1/5. What mistake did they make? Answer in two sentences."}], "max_tokens": 120},
    timeout=300,
)
print(r.status_code)
print(r.json()["choices"][0]["message"]["content"])

## Make it reachable from the internet
This downloads Cloudflare's `cloudflared` and opens a free "quick tunnel" (a random `https://....trycloudflare.com` address that changes every run).
Anyone with the address can *reach* the server, but every request needs the token printed below.

In [ ]:
import re, time

sh("curl -sL -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared")
tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8001"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
deadline = time.time() + 90
while time.time() < deadline and public_url is None:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        public_url = m.group(0)
assert public_url, "Could not get a tunnel URL - re-run this cell."

print("=" * 60)
print("Set these two values in your web app (Render / .env):")
print(f"  LLM_BACKEND_URL={public_url}")
print(f"  LLM_BACKEND_TOKEN={TOKEN}")
print("=" * 60)

In [ ]:
# Keep this cell running while you need the model. Stop it (or close the notebook) to take the model offline.
while True:
    time.sleep(60)